# Semaine 8 — Jour 2 : Benchmarks reproductibles et variabilité

Définir une charge comparable et son protocole de répétition.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour le parcours complet avec entretien et challenge. J1 conserve sa durée et son contenu d’origine; il peut être réparti sur plusieurs sessions.

Prérequis : J1, Python 3.11+, JSON et tests backend. Aucun accès fournisseur, aucune clé LLM et aucune infrastructure externe nécessaires.

Lire [les objectifs](learning_objectives.md), [le cours](chapter.md), puis [le lab](labs/README.md). Tenter [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks : `notebooks/week08/S8_J2_benchmarks.ipynb` et `notebooks/week08/teacher/S8_J2_benchmarks_teacher.ipynb` depuis la racine du dépôt.

Critère de réussite : Distinguer un replay synthétique d’une mesure de capacité en charge.

Les observations et prix sont synthétiques; les résultats mesurent le fonctionnement du protocole local, pas un fournisseur réel.


# Objectifs mesurables

- Définir une charge comparable et son protocole de répétition.
- Calculer p50, p95, maximum et taux d’erreur avec un dénominateur explicite.
- Distinguer un replay synthétique d’une mesure de capacité en charge.

## Preuves attendues

Une sortie reproductible, une interprétation de son dénominateur et une limite opérationnelle correctement expliquée.


# S8 J2 — Benchmarks reproductibles et variabilité


## 1. Évaluer et mesurer ne répondent pas à la même question
Le jour 1 teste un contrat fonctionnel : une réponse, une structure ou une action respecte-t-elle l’attente ? Le benchmark ajoute le temps, la variabilité et les conditions d’exécution. Une version peut être correcte mais trop lente, ou plus rapide tout en violant un refus critique. Les deux familles de preuves doivent rester visibles.

Le fil rouge reprend les douze cas de J1. Chaque cas possède trois essais par version, soit trente-six observations. Les durées et tokens sont fabriqués pour le cours; aucun fournisseur n’a été mesuré. Les répétitions exercent l’agrégateur. Elles ne constituent pas des observations indépendantes de qualité : la sortie de J1 est réutilisée pour les trois essais d’un même cas.

## 2. Fixer le protocole avant les résultats
Décrire le corpus, les identifiants, le nombre de répétitions, la concurrence, le matériel, les paramètres et la politique de cache. Le fichier workload.json conserve une grille de cas et trois répétitions. Son empreinte relie les deux versions au même protocole. Les configurations de modèle diffèrent volontairement; leur empreinte doit être tracée sans interdire la comparaison de versions.

Le code refuse une observation manquante, une paire case_id/repeat dupliquée ou un mélange de configurations dans un même lot. En test réel, consigner les erreurs plutôt que supprimer les requêtes échouées. Un timeout fait partie du comportement du système et du coût de l’utilisateur. Écarter toutes les erreurs réduit artificiellement la latence observée.

## 3. Choisir un quantile et le documenter
Le lab utilise nearest-rank : trier N valeurs et prendre celle de rang plafond(p × N), avec des rangs démarrant à 1. Pour trente-six essais, le p95 est la trente-cinquième valeur. Le maximum reste nécessaire : le candidat possède un timeout à 2200 ms alors que son p95 est de 461 ms. Le p95 ne signifie donc pas « aucune requête au-delà de cette durée ».

La baseline a un p95 de 601 ms. Comparer les mêmes cas donne ici un gain de 140 ms pour le candidat. Ce résultat est une propriété des fixtures. On ne peut pas en déduire qu’un modèle commercial ou une architecture serait plus rapide. Sur de petits effectifs, un quantile évolue par sauts; augmenter le nombre de cas et de répétitions avant de tirer une conclusion générale.

## 4. Mesure séquentielle et charge concurrente
Le débit affiché divise le nombre d’essais par la somme de leurs durées synthétiques. C’est un débit de replay séquentiel, sans coût du client ni du réseau. Ce n’est pas la capacité maximale du serveur. Un test concurrent exige une mesure de temps murale, une politique d’arrivée, le nombre de requêtes en vol et des métriques de saturation.

Distinguer un modèle fermé, où un nombre fixe de clients attend la fin avant de recommencer, d’un modèle ouvert qui génère des arrivées selon un débit cible. Un client saturé peut ne plus produire les arrivées prévues. Un résultat sans explication de la charge ne permet pas d’évaluer ce biais. Le lab ne fournit pas de générateur de charge concurrente.

## 5. Comparaison et décision
Afficher taux d’erreur, effectifs et maximum avec les quantiles. Pour la qualité, conserver une comparaison appariée sur les mêmes cas et inspecter les régressions. Pour des mesures réelles, répartir l’ordre des versions afin de limiter un effet d’échauffement ou de disponibilité du fournisseur. Conserver tous les essais; ne pas publier uniquement le meilleur.

Le candidat de ce cours est plus rapide et moins cher, mais demeure refusé à cause de c09. La règle critique de J1 ne disparaît pas dans une moyenne pondérée vitesse/qualité. Les seuils du projet sont pédagogiques, à justifier pour un besoin réel.

## À retenir
Un benchmark utile contient un protocole reproductible, des observations complètes et des limites. Un p95 faible ne compense pas un échec critique ni un maximum caché.


# Lab exécutable

Cellules autonomes à partir du dépôt; aucune requête réseau.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week08/ops.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
from examples.week08.ops import percentile, benchmark, EvidenceError
from examples.week08.demo import load
assert percentile([100, 200, 300, 1000], .5) == 200
assert percentile([100, 200, 300, 1000], .95) == 1000
data = load("candidate")
ids = [c["id"] for c in data["evaluation"]["cases"]]
result = benchmark(data["runs"], ids)
assert result["count"] == 36 and result["p95_ms"] == 461
assert result["max_ms"] == 2200
print(result)

# Exercices

Conserver une première réponse avant de lire la correction.

## Exercice 1 — Recalculer le p95

Avec [100,200,300,1000], calculer p50 et p95 par nearest-rank.

Livrable : une réponse justifiée, avec calcul ou observation lorsque pertinent.

## Exercice 2 — Inspecter le timeout

Pourquoi le p95 candidat de 461 ms coexiste-t-il avec un maximum de 2200 ms ?

Livrable : une réponse justifiée, avec calcul ou observation lorsque pertinent.

## Exercice 3 — Tester la couverture

Retirer une observation candidate. Pourquoi le lab refuse-t-il le calcul au lieu de produire une moyenne ?

Livrable : une réponse justifiée, avec calcul ou observation lorsque pertinent.


# Entretien

Répondre en deux minutes par question, avec un exemple et une limite.

## Question 1

Pourquoi la moyenne est-elle insuffisante ?

## Question 2

Comment mesurer un vrai gain de capacité ?


# Challenge

Modifier le workload pour exiger quatre répétitions sans ajouter de données. Vérifier le rejet. Proposer ensuite le plan d’un vrai benchmark concurrent, avec une règle d’arrêt et un budget de coût.

## Critères

Reproductibilité, garde vérifiée, distinction entre résultat mesuré et garantie non démontrée.


# Références — J2

Sources de référence consultées le 20 octobre 2026. Le lab implémente ses propres contrats pédagogiques; il n’affirme pas déployer les outils cités.

- [OpenTelemetry : concepts d’observabilité](https://opentelemetry.io/docs/concepts/observability-primer/) — signaux et diagnostic.
- [OpenTelemetry : traces](https://opentelemetry.io/docs/concepts/signals/traces/) — contexte et spans pour une intégration réelle.
- [OWASP : injection de prompt, édition 2025](https://genai.owasp.org/llmrisk/llm01-prompt-injection/) — référence datée, sans prétendre être la dernière édition.
- [OWASP : excessive agency, édition 2025](https://genai.owasp.org/llmrisk/llm062025-excessive-agency/) — permissions et autonomie.
- [Python : Decimal](https://docs.python.org/3/library/decimal.html) — calcul décimal pour les coûts.
- [Python : unittest](https://docs.python.org/3/library/unittest.html) — reproduction des contrôles.

Lire prioritairement les sources liées au thème du jour. Les seuils, tarifs et identités du lab sont fictifs et ne proviennent pas de ces documents. Le calcul nearest-rank est défini explicitement dans le chapitre et le code.
